# 01 — Feature engineering

**Début** : charge `00_pull_data`. **Fin** : sauvegarde metadata + keywords.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()

sys.path.insert(0, str(REPO_ROOT))


In [2]:

import json

from shared.step_artifacts import load_previous_step_output, rel_path, resolve_path, save_step_output

prev = load_previous_step_output("01_feature_engineering")
cards_dir = resolve_path(prev["cards_dir"])
print(f"Dataset (étape 00): {cards_dir}")

CTK = REPO_ROOT / "clean-text-to-keywords"
sys.path.insert(0, str(CTK))

from keyword_extractor import KeywordExtractor
from json_inference import fill_template_from_keywords

extractor = KeywordExtractor.from_default_model()


[00_pull_data] État chargé ← /home/light/Documents/DO5/MLOps/mlops/artifacts/pipeline/steps/00_pull_data/state.json
Dataset (étape 00): /home/light/Documents/DO5/MLOps/mlops-pokegen/data/raw/cards


In [3]:
user_prompt = "A fierce fire dragon pokemon with wings and a blazing tail attack"
keywords = extractor.extract(user_prompt)
print("Keywords:", keywords)

with open(CTK / "json_template_example.json", encoding="utf-8") as f:
    template = json.load(f)

meta = fill_template_from_keywords(template, keywords)
print(json.dumps(meta, indent=2, ensure_ascii=False)[:1500])


Keywords: ['fierce', 'fire', 'dragon', 'pokemon', 'flying', 'tail', 'attack']
{
  "category": "Pokemon",
  "name": "Pokemon",
  "rarity": "",
  "hp": "70",
  "types": [
    "fire",
    "dragon"
  ],
  "evolveFrom": "",
  "description": "Pokemon is a fire-type Pokemon often found in unknown. It commonly uses tail, attack and shows abilities like balanced adaptation.",
  "stage": "Basic",
  "attacks": [
    {
      "cost": [
        "Fire"
      ],
      "name": "Tail",
      "effect": "Deals damage with tail."
    },
    {
      "cost": [
        "Fire"
      ],
      "name": "Attack",
      "effect": "Deals damage with attack.",
      "damage": 40
    }
  ],
  "weaknesses": [
    {
      "type": "water",
      "value": "x2"
    }
  ],
  "retreat": 2,
  "regulationMark": "",
  "legal": {
    "standard": true,
    "expanded": true
  }
}


In [4]:
from shared.step_artifacts import step_dir

out_dir = step_dir("01_feature_engineering")
out_dir.mkdir(parents=True, exist_ok=True)
metadata_path = out_dir / "example_metadata.json"
metadata_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

save_step_output("01_feature_engineering", {
    "cards_dir": prev["cards_dir"],
    "user_prompt": user_prompt,
    "keywords": keywords,
    "metadata_path": rel_path(metadata_path),
})


[01_feature_engineering] État sauvegardé → /home/light/Documents/DO5/MLOps/mlops/artifacts/pipeline/steps/01_feature_engineering/state.json


PosixPath('/home/light/Documents/DO5/MLOps/mlops/artifacts/pipeline/steps/01_feature_engineering/state.json')